# 33 · RAG 重做：更多资料、完整流程、分层评测

## 这一章在整个项目里的位置

```
00 总览 → … → 18 疾病介绍（RAG） ✅ → … → 30 离线试打分 ✅ → 31 加进 Orphanet 的病 ✅ → 32 不花钱的改进 ✅ → [33 RAG 重做 ✅] → 34 似然比打分和亚型折叠 → 35 固定流程和试跑
```

> **这一章的代码在分支 `feat/rag-redo` 里。运行这本笔记不调用任何大模型**，约 10 分钟。建索引（算向量）一共花了约 3 小时 CPU，已经做完；下面重新运行只会确认"已经建好"。要先打开 Docker 里的 Qdrant 服务器（准备那一步讲）。

## 背景：为什么重做

第 18 章做过一版 RAG，第 20 章拿掉它考了一次：**几乎不掉分**（41.5% → 40.5%，4 比 2），还多花 18% 的 token，于是从 agent 里去掉了。

你问了三个问题：**是不是资料不全？为什么不用企业标准的做法？RAG 怎么评测？** 这一章逐个回答：

| 第 18 章 | 这一章 |
|---|---|
| 两个来源：Orphanet 定义（中位 51 个词）、MedlinePlus | 五个来源：加上 **GeneReviews**、**Wikipedia**、Mondo |
| 只有 53% 的病有介绍 | 77% |
| 一种切法 | 比了五种切法 |
| 一个向量模型 | 比了三个 |
| 没有重排、没有过滤 | 试了重排；"只查这个病"的过滤 |
| 只量"按症状找病" | 三类题：找病、查遗传方式、查有没有某个症状 |
| 遮挡：只能遮整个病 | 还能按段落遮：引用了病人自己论文的段落不返回 |

## RAG 的完整流程（公司里的标准做法）

```
① 收集资料 → ② 清洗 → ③ 切块 → ④ 向量化 → ⑤ 存进向量库 → ⑥ 检索（+过滤） → ⑦ 重排 → ⑧ 交给模型回答、标出处
                                                     ↑ 评测第一层：搜得准不准（不花钱）        ↑ 第二层：答得对不对（花钱）
```

这一章把每一步都做一遍，**每一步都比几种做法，用数据选**。

---

## 准备

运行下面这一格，定义执行命令的小工具 `run`。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat", COLUMNS="160")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
NOISE = ("INFO mlflow", "WARNING mlflow", "Downloading artifacts", "Fetching ")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print("\n".join(line for line in out.splitlines() if not any(n in line for n in NOISE)))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

再确认向量库服务器开着。第一次要先启动 Docker（我们这台电脑用 colima）和 Qdrant 容器：

```
colima start
docker start raredx-qdrant      # 第一次是 docker run，见第 4 步
```

In [ ]:
run("curl -s localhost:6333")

看到 `"version":"1.19.2"` 就是开着的。

---

## 第 1 步 · 先给旧版打分：对照组

### 在干什么
用这一章新出的考题，先考第 18 章的旧索引。后面每一个改进，都和它比。

### 背景：考题怎么出
都从开发病人里来（dev-200、HMS、RAMEDIS-200、MME、病例报告），每道题照常遮挡病人自己的论文。三类：

| 题型 | 例子 | 标准答案从哪来 | 量什么 |
|---|---|---|---|
| **按症状找病**（1,022 道） | "Seizure; Intellectual disability; …" | 病人的确诊 | 搜出来的段落讲的病里，前 10 个有没有正确答案 |
| **查遗传方式**（337 道） | "Phenylalanine hydroxylase deficiency: mode of inheritance" | Orphanet 的结构化数据 | 前 5 段里，有没有一段**讲这个病、而且写出了答案** |
| **查有没有某个症状**（1,479 道） | "Cardiomyopathy, dilated, 1A: Ventricular arrhythmia" | HPO 的注释 | 同上 |

关键的一点：**标准答案不是从被检索的资料里来的**，而是来自另外的结构化数据（Orphanet 的遗传方式表、HPO 的注释）。这样不用人标注，几千道题也能出；而且不会"拿资料考资料"。

"写出了答案"是机械判断：段落里出现"autosomal recessive"这样的字样，或者症状的名字和同义词。换一种说法写的会被漏判，所以**这些数是下限**。

### 指标的名字，对应公司里常说的

| 我们的 | 意思 | 公司里常用的叫法（比如 Ragas 这个评测库） |
|---|---|---|
| about@5 | 前 5 段里有讲这个病的 | 命中率（hit rate） |
| answer@5 | 前 5 段里有写出答案的 | 上下文召回（context recall） |
| rr | 第一段写出答案的排第几，取倒数（第 1 名 = 1，第 2 名 = 0.5） | MRR |

### 执行
约 1 分半：

In [ ]:
run("uv run raredx eval rag --index chapter18 --threads 2")

### 结果怎么看
- **按症状找病**：前 10 个病里有正确答案的只有 **14.3%**
- **查遗传方式**：前 5 段里有讲这个病的占 62.6%，**写出答案的只有 10.7%**
- **查症状**：同样，42.2% 对 9.3%

**旧版的问题一目了然：搜得到讲这个病的段落，但段落里不写答案。**Orphanet 的定义只有 50 个词；MedlinePlus 我们当时只收了"描述"那部分，讲遗传方式的段落根本没收。

### 面试怎么说
> "Before improving the RAG I scored the old one on new questions with answer keys taken from structured data, not from the texts: Orphanet's inheritance table and HPO's annotations. It found a passage about the right disease 63% of the time, but one saying the answer only 11% — the texts were too thin, not the retrieval."

---

## 第 2 步 · 多拿资料

### 在干什么
加三个来源，把五个来源读成同一种格式：**文档 → 章节 → 段落，每段记下它引用了哪些论文**。

### 用了什么资料，为什么

| 来源 | 内容 | 许可 | 为什么要 |
|---|---|---|---|
| **GeneReviews** | 888 章，每章有诊断、临床表现、鉴别诊断、遗传咨询；每段都标了引用的论文 | **非商用研究**；不能改了再发布，所以只放本机、不进 git | 写得最详细的临床资料，正好补遗传方式、典型表现 |
| **Wikipedia** | 2,849 篇讲知识库里的病的文章 | CC BY-SA，注明出处 | "相对常见"的罕见病写得多（白塞病、军团病） |
| Orphanet（补全） | 全部 5,082 个病的定义（第 18 章只收了能对上 OMIM 的） | CC BY 4.0 | 第 31 章合并进来的病也有介绍 |
| Mondo | 6,766 条疾病定义 | CC BY 4.0 | 覆盖面广 |
| MedlinePlus | 原有的 | 公有领域 | — |

没用的：OMIM 全文（质量最高，但要用你的邮箱注册、条款限制多）。

### Wikipedia 怎么拿的，以及两个插曲
1. 在 **Wikidata** 里找所有登记了 Orphanet 或 OMIM 编号的英文文章：14,274 篇，其中讲知识库里的病的 2,903 篇（其余多是讲基因的）
2. 记下每篇文章**此刻的版本号**，清单进 git（`evals/wikipedia_revisions.tsv`）
3. 按版本号下载。文章明天被人改了，我们读到的还是今天这一版：**可复现**

**插曲一**：第一次一个接一个不停地请求，被 Wikipedia 限速（返回 429 "请求太多"）。改成每篇之间停 1 秒、收到 429 就按它说的时间等待再试。**调用外部接口都要这样：限速 + 退避重试。**

**插曲二**：我在第一个抓取还没停的时候又开了第二个，两个进程同时往一个文件里追加，把文件写坏了。好在每篇是单独压缩的，救回了 2,772 篇，剩下的重新抓。**一个文件只能有一个写的人。**

最后发现 37 篇文章被两个旧标题共用（比如"鱼眼病"和"卵磷脂胆固醇酰基转移酶缺乏症"是同一篇），合并后 2,849 篇。

### 公司里什么情况怎么做
- 资料先登记来源、版本、许可、校验码（`evals/datasets.toml`），写清楚能做什么、不能做什么
- 网页类资料按版本固定；做成可以断点续抓

### 执行
把五个来源读成文档（约 1 分钟，不联网）：

In [ ]:
run("uv run raredx data corpus")

### 结果怎么看
- GeneReviews 888 章、330 万词；Wikipedia 2,849 篇、360 万词；另外三个来源都是短定义
- **知识库 10,711 个病里，8,294 个有至少一份介绍：77%**。第 18 章是 4,493 个（按现在的 10,711 个病算 42%；按当时只读 OMIM 的 8,478 个病算 53%）

看一章 GeneReviews 读出来的样子：

In [ ]:
import json
from raredx.knowledge.corpus import Document
folder = next((ROOT / "data/processed/rag-corpus").iterdir())
docs = {d.id: d for d in (Document.model_validate_json(l) for l in open(folder / "documents.jsonl"))}
pku = docs["GeneReviews:pku"]
print(pku.title, pku.disease_ids, pku.url)
for s in pku.sections[:8]:
    p = s.paragraphs[0]
    print(f"  [{s.title}]  {p.text[:90]}...  引用 {p.pmids[:3]}")

- 每一节都有标题，每段都知道自己引用了哪几篇论文。治疗、分子遗传、参考文献这些和诊断无关的节，在清洗时去掉了

### 面试怎么说
> "I added GeneReviews, Wikipedia and Mondo to the corpus, each registered with licence and checksum; Wikipedia articles are pinned by revision id, so the corpus is reproducible. Every source is parsed into documents, sections and paragraphs, each paragraph keeping the PubMed ids it cites — which is what lets the evaluation mask a case's own paper paragraph by paragraph. Coverage went from 42% to 77% of the knowledge's diseases."

---

## 第 3 步 · 切块：五种切法

### 在干什么
同一份文档，用五种方法切成块，后面比哪种搜得最准。

### 五种切法

| 切法 | 怎么切 | 打个比方 |
|---|---|---|
| **固定长度** | 每 1,000 个字符一块，前后重叠 150，不管结构 | 把书每 10 页撕一沓，不管章节 |
| **递归** | LangChain 的 `RecursiveCharacterTextSplitter`：先按段落切，太长再按句子、按词。**公司里最常用的默认做法** | 尽量在段落处撕 |
| **按章节** | 只在同一节里装块，一块不会跨两节 | 每一章单独撕 |
| **按章节 + 上下文** | 同上，每块前面加"病名 — 小节名：" | 每一沓上贴个标签 |
| **父子块** | 用小块（约 300 字符）去搜，命中后返回它所在的大块 | 按索引卡找到，再读整页 |

### 执行
看同一段话在五种切法下变成什么样：

In [ ]:
from raredx.knowledge import chunking
for s in chunking.STRATEGIES:
    chunks = chunking.chunk(pku, s)
    c = next(c for c in chunks if "autosomal recessive" in c.text)
    print(f"{s:16} {len(chunks):4} 块 | {c.text[:110]!r}")

### 结果怎么看
- **固定长度**和**递归**切出来的块，开头可能是半句话，也不知道属于哪一节
- **按章节 + 上下文**的块，开头就写着"Phenylalanine Hydroxylase Deficiency — Summary — Genetic counseling:"，单独拿出来也知道在讲什么

### 公司里什么情况怎么做
- 文档有结构（章节、段落）就按结构切，再加上下文标题（Anthropic 把这叫 Contextual Retrieval）
- 切法要用数据比，下一步就比

---

## 第 4 步 · 向量化，存进向量库服务器

### 在干什么
给每一块算向量，存进 Qdrant。数据大了以后，从"程序里的内存模式"换成"独立的 Qdrant 服务器"。

### 背景
第 18 章的向量库在程序内存里，7,790 块没问题。这一章全部资料有 **91,765 块**，还要按条件过滤（只看某个病、遮挡某篇论文）。我在 1 万块上量过：

| | 内存模式 | Qdrant 服务器（Docker） |
|---|---|---|
| 不过滤 | 88 毫秒 | **7 毫秒** |
| 遮挡某篇论文 | 163 毫秒 | **10 毫秒** |
| 只看某个病 | 159 毫秒 | **8 毫秒** |

服务器给向量建了索引（HNSW），也给"病""论文"这两个过滤字段建了索引，所以快十几倍。**代码一样，只改一个地址**（`settings.qdrant_url`）。

启动服务器的命令（已经做过）：
```
docker run -d --name raredx-qdrant -p 6333:6333 -v raredx-qdrant:/qdrant/storage qdrant/qdrant:v1.19.2
```
版本号写死（1.19.2，和我们的客户端 1.19 对得上）；数据放在 Docker 自己的卷里。

### 一个坑：同一个评测跑两次，结果要一样
一开始，同一个评测跑两次，数字差零点几个点。查下来：只按向量、只按关键词搜，两次完全一样；**混合搜索合并两份名单时，分数打平的段落谁排前面，服务器每次可能不同**（它把数据分几块并行地搜）。解决办法：两份名单还是让服务器出，合并这一步放在我们自己的代码里，用同一个公式，分数打平就按编号排。现在跑两次完全一样。另外向量搜索用"精确"模式（逐个比对），不用"近似"模式：9 万块精确搜也只要几毫秒。**评测结果不能复现，就没法比较。**

### 算向量的速度，以及一次教训
这台 M4 上，BGE-small 每秒约 25 块，BGE-base 约 8 块，BGE-large 约 5 块。全部资料用 BGE-small 也要 70 分钟。

**教训**：我一开始想"用满 10 个核"，开了 10 个进程，每个都加载一份模型，内存被占满，电脑卡住，进程被系统杀掉。之后建索引都只用 5–6 个核（`--threads`），留一半给人用。

### 执行
建索引的命令（已经建好的会直接跳过）：

In [ ]:
for name, strategy in [("gr300-fixed", "fixed"), ("gr300-recursive", "recursive"), ("gr300-section", "section"),
                       ("gr300-section-context", "section+context"), ("gr300-parent", "parent")]:
    run(f'uv run raredx data rag-index {name} --strategy "{strategy}" --source genereviews --chapters 300')
run("uv run raredx data rag-index gr300-section-context-bge-base --strategy section+context --model BAAI/bge-base-en-v1.5 --source genereviews --chapters 300")
run("uv run raredx data rag-index gr300-section-context-arctic-m --strategy section+context --model snowflake/snowflake-arctic-embed-m --source genereviews --chapters 300")
run("uv run raredx data rag-index all-section-context --strategy section+context")

### 结果怎么看
- `gr300-…`：比较切法和向量模型用的**样本**：考题涉及的病对应的 158 章 GeneReviews，再随机加 142 章当干扰，共 300 章。用样本是因为算力：五种切法全量要 24 万块、三个多小时
- `all-section-context`：全部五个来源，用最后选定的切法，91,765 块

### 面试怎么说
> "At 90,000 chunks with metadata filters, Qdrant's in-process mode took about 160 ms a query; the same code against a Qdrant server in Docker took 10 ms, with payload indexes on the disease and paper fields. Embedding on a laptop CPU is the real bottleneck — 25 chunks a second — so model and chunking comparisons ran on a 300-chapter sample."

---

## 第 5 步 · 检索评测：一样一样比

### 在干什么
在 GeneReviews 样本上，只考样本里有资料的病（944 道题），比：切法、检索方式、过滤、重排、向量模型。

**这些都已经考过、记在 MLflow 的 `rag-retrieval` 实验里**（在 MLflow 网页上可以并排看），下面直接读出来，不重跑。公司里也是这样：实验跑一次、记下来，看结果从记录里读。

### 执行

In [ ]:
run("uv run raredx eval rag-runs --prefix gr300-")

### 结果怎么看：一样一样看

**① 切法**（`… | hybrid` 那几行）
- 查症状：**按章节 + 上下文 29.3%** 最好，固定长度 25.2%、递归 23.5%
- 查遗传方式："答案排多靠前"（rr）上，按章节类的 69–73% 远好于固定长度、递归的 52%：标题让"遗传咨询"那一节排到了第一
- 按症状找病：固定长度 58.2% 反而最高。下面用逐题配对检验看它是不是真的

**② 检索方式**（`gr300-section-context | dense / keyword / hybrid`）
- 按症状找病：只按意思（向量）46.7%，只按关键词 52.7%，**混合 56.6%**
- 罕见病里专有名词多，只靠向量会把它们冲淡；混合检索最稳

**③ 过滤：只查这个病**（`… filtered` 那几行）
- 查遗传方式：80.0% → **92.7%**；查症状：29.3% → 32.9%
- **知道要问哪个病，就该过滤，不要靠相似度去碰运气**。这是公司里的标准做法：按元数据过滤（权限控制也是这么做的）

**④ 重排**（`… Xenova/ms-marco-MiniLM-L-6-v2`）：先捞 30 段，再用一个专门判断相关度的小模型逐段打分
- 过滤后，查遗传方式 92.7% → 95.2%，答案排第一的更多（rr 85.2% → 90.7%）；查症状 32.9% → 31.5%，没帮上
- 不过滤时，没帮助。大的重排模型（BGE-reranker）在 Mac 上每道题要 11 秒，没法试，等以后租了卡再说

**⑤ 向量模型**（`… -bge-base`、`… -arctic-m`）
- BGE-base（大 3 倍、慢 3 倍）没有更好；Snowflake arctic-m 普遍更差

### 逐题配对检验：哪些差别是真的

In [ ]:
run('uv run raredx eval rag-compare --run "gr300-section-context | hybrid" --run "gr300-fixed | hybrid"')
run('uv run raredx eval rag-compare --run "gr300-section-context | hybrid filtered" --run "gr300-section-context | hybrid" --measure feature:answer@5 --measure inheritance:answer@5')
run('uv run raredx eval rag-compare --run "gr300-section-context-bge-base | hybrid" --run "gr300-section-context | hybrid"')
run('uv run raredx eval rag-compare --run "gr300-section-context-bge-base | hybrid filtered" --run "gr300-section-context | hybrid filtered" --measure feature:answer@5 --measure inheritance:answer@5')

- **按章节 + 上下文 vs 固定长度**：查症状 23 : 6（p = 0.002），**真的更好**；查遗传方式，前 5 段有没有答案打平（8 : 9），但答案排得更靠前（67 : 14，显著）；按症状找病 21 : 27（p = 0.47），固定长度那 1–2 个点的领先是噪声
- **过滤 vs 不过滤**：查遗传方式 21 : 0、查症状 15 : 0，**过滤显著更好**，而且一道都没变差
- **BGE-base vs BGE-small**：按症状找病 26 : 13（p = 0.053，不显著）；过滤后查症状 0 : 7（p = 0.016），**大的反而差**

**决定**：按章节 + 上下文；混合检索；查某个病的资料时过滤；向量模型留 BGE-small；重排做成可选，默认不开。

### 公司里什么情况怎么做
- 一次只改一样，其他不变，才知道是哪一样起的作用
- 数字差一两个点时，**先做配对检验再下结论**
- 更大的模型不一定更好，还更慢、更贵：用数据选

### 面试怎么说
> "On a 300-chapter sample I varied one thing at a time — five chunkings, three retrieval modes, a metadata filter, a reranker, three embedding models — and tested each difference question by question. Section-aware chunks with a contextual header beat fixed-size chunks on feature questions, 23 to 6; filtering to the disease asked about helped every question it changed, 21 to 0 on inheritance; a three-times-larger embedding model was not better."

---

## 第 6 步 · 全部资料，和第 18 章比

### 在干什么
用选定的做法，在全部五个来源上考全部 2,838 道题，和第 1 步的旧版逐题比。

### 执行
约 3 分钟：

In [ ]:
run("uv run raredx eval rag --index all-section-context --threads 2")
run('uv run raredx eval rag-compare --run "all-section-context | hybrid" --run "chapter18 | hybrid"')
run('uv run raredx eval rag-compare --run "all-section-context | hybrid filtered" --run "chapter18 | hybrid filtered" --measure feature:answer@5 --measure inheritance:answer@5')

### 结果怎么看

| | 第 18 章 | 新的（全部资料） | 逐题比 |
|---|---|---|---|
| 按症状找病：前 10 个病里有正确答案 | 14.3% | **28.9%** | 176 : 27 |
| 查遗传方式：前 5 段写出答案（过滤） | 13.6% | **63.5%** | 169 : 1 |
| 查症状：前 5 段写出答案（过滤） | 10.0% | **32.3%** | 331 : 2 |

- **每一项都显著更好**，几乎没有哪道题变差
- 但要看清楚两点：
  - **按症状找病仍然远不如按症状排序的工具**：dev-200 上 23.6%，排序工具前 10 名是 49.5%（第 32 章）。疾病介绍讲的是"一类病"，分不清亚型。**RAG 不适合当提名工具**
  - **查遗传方式在全部资料上 63.5%，样本上是 92.7%**：GeneReviews 一定写遗传方式，其他来源（Orphanet、Mondo 的一两句定义）多半不写。病例报告那一类只有 50.3%：这些病很多不是单基因病，GeneReviews 不收

**结论：RAG 的用处在"查候选病的资料来核对"，不在"找病"**。下一章的固定流程里，它放在"核对"那一步：拿候选病名去查遗传方式、发病年龄、典型表现。

### 面试怎么说
> "On all 2,838 questions the rebuilt RAG beat the old one on every measure — inheritance found in the top five 64% against 14%, 169 questions to 1. But finding the disease from symptoms stayed far below the structured phenotype ranking, 24% against 50% on the same cases: descriptions are written per disease family. So in the agent the RAG moves from nominating candidates to checking them."

---

## 第 7 步 · 防泄露：按段落遮挡

### 在干什么
数一数：如果只遮"整个病"、不按论文遮段落，会有多少答案漏进来。

### 背景
第 18 章的资料不注明出处，只能遮"只从这篇论文知道的病"。GeneReviews 和 Wikipedia 每段都标了引用，所以能做得更细：**引用了病人自己论文的段落，检索时直接过滤掉**（在 Qdrant 里，和权限控制一个做法）。

### 执行

In [ ]:
from collections import Counter
from raredx import cli
from raredx.config import settings
questions = [q for q in cli._rag_questions(settings.evals_dir, settings.data_dir) if q.kind == "find"]
cites = {}
for line in open(ROOT / "data/processed/rag-index/all-section-context/passages.jsonl"):
    p = json.loads(line)
    for pmid in p["pmids"]:
        cites.setdefault(pmid, []).append((p["doc"].split(":")[0], set(p["disease_ids"])))
for source in ("dev-200", "rarearena-dev"):
    mine = [q for q in questions if q.source == source and q.hide_pmids]
    leaking = [q for q in mine if any(set(q.accepted) & d for _, d in cites.get(q.hide_pmids[0], []))]
    by = Counter(s for q in mine for s, _ in cites.get(q.hide_pmids[0], []))
    print(f"{source}: {len(mine)} 道题；有讲正确答案的段落引用了病人自己论文的：{len(leaking)} 道（{len(leaking)/len(mine):.0%}）；这些段落来自 {dict(by)}")

### 结果怎么看
- **dev-200：15% 的题**，资料里有讲正确答案、又引用了这道题自己论文的段落，大多来自 GeneReviews。不按段落遮，这些题的答案就漏进来了
- 病例报告那一类只有 1%：病例报告很少被 GeneReviews、Wikipedia 引用

### 公司里什么情况怎么做
- **泄露检查要量出来**，不能只说"我们遮了"
- 资料带出处，就按出处过滤；过滤放在检索这一层，任何调用都绕不过去

### 面试怎么说
> "Because GeneReviews and Wikipedia cite papers paragraph by paragraph, masking moved from whole diseases to paragraphs: a chunk citing the case's own paper is excluded by a filter in the vector store. Without it, 15% of the Phenopacket cases would have had a passage about the right answer resting on their own paper."

---

## 第 8 步 · 评测第二层：模型答得对不对（花钱，待你同意）

### 在干什么
同一批"遗传方式"的题，问模型两遍：
- **闭卷**：只凭它自己知道的
- **开卷（RAG）**：给它第 6 步查到的 5 段资料，只许照资料答，资料没写就答"not stated"，还要说明是根据哪几段答的

判分**用程序，不用另一个模型当评委**：答案是几个固定选项之一，直接和 Orphanet 比对；开卷的答案，它引用的段落里要真的写着这个答案，才算"有依据"。

### 公司里什么情况怎么做
- RAG 的第二层通常量三样：**答得对不对**、**有没有依据**（每句话资料里都有）、**资料没写时会不会老实说不知道**
- 答案是自由文字时，常用一个模型当评委（MLflow、Ragas 都内置）；**能用程序判的，就不用模型判**：程序不会被说服，也不花钱

### 执行
命令（调用模型，100 道题约 ¥0.15–0.3，设了 $0.20 的上限）：

```
uv run raredx eval rag-answers --index all-section-context --n 100
```

跑过以后结果记在 MLflow 的 `rag-answers` 实验里，下面这一格把最近一次读出来：

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["rag-answers"], order_by=["start_time DESC"], max_results=1) if mlflow.get_experiment_by_name("rag-answers") else []
if len(runs) == 0:
    print("还没跑：要你同意后才调用模型")
else:
    r = runs.iloc[0]
    for book in ("closed", "open"):
        print(book, {m: f"{r[f'metrics.{book}.{m}']:.1%}" for m in ("right", "wrong", "not_stated")})
    print("开卷答案有依据：", f"{r['metrics.open.grounded']:.1%}", "；资料里写着答案：", f"{r['metrics.passages_say_it']:.1%}", "；花费 $", round(r["metrics.cost_usd"], 4))

---

## 第 9 步 · 结论

| 决定 | 依据 |
|---|---|
| 资料：五个来源，有介绍的病 42% → 77% | 第 2 步 |
| 切法：按章节 + 上下文 | 第 5 步：查症状显著好于固定长度 |
| 检索：混合；查某个病时过滤 | 第 5 步：过滤一道题都没变差 |
| 向量模型：BGE-small | 第 5 步：大模型没有更好 |
| 重排：可选，默认不开 | 第 5 步：只在一项上有小帮助 |
| **RAG 的用处：核对候选，不是提名** | 第 6 步：找病远不如症状排序；查资料大幅提升 |
| 遮挡：按段落 | 第 7 步：否则 dev-200 有 15% 的题会漏 |

**工具也更新了**：`search_disease_texts` 多了一个可选参数 `disease_id`（只查这个病），遮挡改成在向量库里过滤，命中小块时返回它所在的大块。它要不要回到 agent 里、放在哪一步，等下一章的固定流程、在付费运行里量。

---

## 本章小结

| 步骤 | 用的技术 | 在公司里的作用 |
|---|---|---|
| 1 对照组 | 结构化数据当标准答案；三类题 | 先量旧版，后面才有比较 |
| 2 多拿资料 | 登记许可、按版本固定、限速重试、断点续抓 | 资料决定 RAG 的上限 |
| 3 切块 | 五种切法；上下文标题 | 有结构就按结构切 |
| 4 向量库 | Qdrant 服务器（Docker）、过滤字段索引 | 数据大了换服务器，代码不变 |
| 5 检索评测 | 命中率、上下文召回、MRR；一次改一样；配对检验 | 用数据选每一步 |
| 6 和旧版比 | 全量、逐题比 | 改进要量出来 |
| 7 防泄露 | 按段落的出处过滤 | 泄露要量出来 |
| 8 生成评测 | 闭卷 vs 开卷；程序判分 | 第二层评测 |

**一句话**：资料从 2 个来源扩到 5 个，有介绍的病 42% → 77%；按公司的标准流程每一步都比过，选出"按章节 + 上下文、混合检索、按病过滤"；查遗传方式的答案出现在前 5 段的比例从 14% 到 64%。但 RAG 找病仍然远不如症状排序，所以它的位置是核对候选。

## 面试题

> **Q: Your first RAG didn't help the agent. Did you give up on RAG?**
>
> A: "No — I measured why. Scored on questions with answer keys from structured data, the old corpus found the right disease's passage but rarely one saying the answer: the texts were thin. I added GeneReviews, Wikipedia and Mondo, compared chunkings, retrieval modes, filtering, reranking and embedding models one at a time, and the answer-bearing rate for inheritance went from 14% to 64%. Finding diseases from symptoms stayed weak, so the RAG's job became checking candidates, not nominating them."

> **Q: How do you evaluate a RAG system?**
>
> A: "In layers. Retrieval first, without a model: hit rate, context recall and MRR on questions whose answers come from somewhere other than the corpus. Then generation: the same questions closed-book and open-book, scored by code where the answer is categorical — right, grounded in a cited passage, or honestly 'not stated'. Then the end task, the agent's diagnosis, in a paid run."

> **Q: How do you stop a RAG tool from leaking the answer in evaluation?**
>
> A: "Every chunk carries the papers its paragraphs cite, and the vector store filters out any chunk citing the case's own paper. I measured what that prevents: 15% of the Phenopacket cases had such a passage about the right answer."

## 下一章

**34 · 似然比打分和亚型折叠**（不花钱）：第 32 章里设计 S1 剩下的两样。

**35 · 固定流程和 ci-20 试跑**：RAG 放在"核对"那一步。